# 05 - Memory Store

Demonstrates how to use the AIMU `MemoryStore` class for storing and retrieving factual memories in subject-predicate-object form using ChromaDB as the vector store backend.

## A - Basic Setup

`MemoryStore` stores facts as natural-language strings in subject-predicate-object form (e.g. `"Paul works at Google"`). By default it uses an in-memory ephemeral store; pass `persist_path` to save facts to disk across sessions.

In [ ]:
from aimu.memory import MemoryStore

# In-memory store (ephemeral — facts are lost when the process exits)
store = MemoryStore()

print(f"Store created. Facts stored: {len(store)}")

## B - Storing Facts

Use `store_fact()` to add a fact string. The string should follow subject-predicate-object form, but any natural-language sentence works — ChromaDB embeds it for semantic retrieval.

In [ ]:
# Store facts about people
store.store_fact("Paul works at Google")
store.store_fact("Paul is married to Sarah")
store.store_fact("Paul enjoys hiking on weekends")
store.store_fact("Sarah is the sister of Emma")
store.store_fact("Emma lives in Paris")
store.store_fact("Emma teaches mathematics at a university")
store.store_fact("Sarah volunteers at a local animal shelter")

print(f"Facts stored: {len(store)}")

## C - Retrieving Facts by Topic

`retrieve_facts(topic)` uses ChromaDB's vector similarity search to find facts semantically related to the topic. This means a query like `"employment"` will match `"Paul works at Google"` even though the word "employment" doesn't appear in the fact.

In [ ]:
print("=== Topic: 'work and employment' ===")
for fact in store.retrieve_facts("work and employment"):
    print(f"  {fact}")

print()
print("=== Topic: 'family relationships' ===")
for fact in store.retrieve_facts("family relationships"):
    print(f"  {fact}")

print()
print("=== Topic: 'hobbies and leisure' ===")
for fact in store.retrieve_facts("hobbies and leisure"):
    print(f"  {fact}")

## D - Limiting the Number of Results

Pass `n_results` to cap how many facts are returned. Results are always ordered by relevance (highest similarity first).

In [ ]:
# Return only the top 2 most relevant facts for a topic
top_facts = store.retrieve_facts("people and where they live or work", n_results=2)

print("Top 2 facts about people's locations/workplaces:")
for fact in top_facts:
    print(f"  {fact}")

## E - Deleting Facts

`delete_fact()` removes all stored facts that exactly match the given string.

In [ ]:
print(f"Before deletion: {len(store)} facts")

store.delete_fact("Emma lives in Paris")

print(f"After deletion:  {len(store)} facts")

# Confirm it's gone
results = store.retrieve_facts("Paris")
print(f"\nSearch for 'Paris' after deletion: {results}")

## F - Persistent Storage

Pass a `persist_path` to save facts to disk. The store survives process restarts and can be shared across sessions.

In [ ]:
import tempfile
import os

# Use a temp directory so this notebook is self-contained
persist_dir = tempfile.mkdtemp(prefix="memory_store_")

# Session 1 — populate
store1 = MemoryStore(persist_path=persist_dir)
store1.store_fact("Paul works at Google")
store1.store_fact("Sarah is the sister of Emma")
print(f"Session 1: stored {len(store1)} facts to {persist_dir}")

# Session 2 — reload from disk (simulating a new process)
store2 = MemoryStore(persist_path=persist_dir)
print(f"Session 2: loaded {len(store2)} facts from disk")
for fact in store2.retrieve_facts("people"):
    print(f"  {fact}")

# Clean up
import shutil
shutil.rmtree(persist_dir)

## G - Using MemoryStore with a Model Client

A natural integration pattern: retrieve relevant facts from the memory store and inject them into the model's system prompt before a conversation, giving the model grounded knowledge about the user or domain.

In [ ]:
# Retrieve facts relevant to an upcoming conversation topic
topic = "Paul's professional background"
relevant_facts = store.retrieve_facts(topic, n_results=3)

# Build a system prompt grounded in stored facts
facts_block = "\n".join(f"- {f}" for f in relevant_facts)
system_prompt = f"""You are a helpful assistant. Here are some facts you know about the user:

{facts_block}

Use these facts to personalise your responses."""

print("Generated system prompt:")
print(system_prompt)

# To use with a model client:
# from aimu.models import OllamaClient as ModelClient
# model_client = ModelClient(ModelClient.MODELS.LLAMA_3_1_8B)
# model_client.system_message = system_prompt
# response = model_client.chat("What do you know about my job?")